# 08 - AgentCore Gateway
Create an MCP Gateway, add Lambda and agent tool targets, and invoke tools through the gateway using the MCP protocol.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
import time
from pathlib import Path
from cleanup_utils import ResourceTracker

config = json.loads(Path("config.json").read_text())
REGION = config["REGION"]
PREFIX = config["PROJECT_PREFIX"]
ACCOUNT_ID = config["ACCOUNT_ID"]

tracker = ResourceTracker()
print(f"\u2713 Config loaded (region={REGION}, prefix={PREFIX})")

## Step 2: Initialize AgentCore Gateway Client

In [ ]:
from bedrock_agentcore.runtime import BedrockAgentCoreApp

# Use boto3 for the gateway management API
agentcore = boto3.client("bedrock-agentcore", region_name=REGION)

print("\u2713 AgentCore client initialized")

## Step 3: Create MCP Gateway

The MCP Gateway provides a centralized tool registry that agents can discover and invoke.

In [ ]:
GATEWAY_NAME = f"{PREFIX}-mcp-gateway"

# Check if gateway exists
existing_gateways = agentcore.list_gateways()
existing = next(
    (g for g in existing_gateways.get("gateways", []) if g["name"] == GATEWAY_NAME),
    None
)

if existing:
    gateway_id = existing["id"]
    print(f"\u2713 Gateway already exists: {gateway_id}")
else:
    print(f"\u231b Creating MCP Gateway: {GATEWAY_NAME}...")
    response = agentcore.create_gateway(
        name=GATEWAY_NAME,
        description="OrgIQ MCP Gateway for centralized tool management",
        protocolType="MCP",
        tags={"project": PREFIX}
    )
    gateway_id = response["id"]
    tracker.add_agentcore_gateway(gateway_id, GATEWAY_NAME)
    print(f"\u2713 Gateway created: {gateway_id}")

## Step 4: Create a Lambda Tool Target

First, create a simple Lambda function that the gateway can invoke.

In [ ]:
import zipfile
import io

lambda_client = boto3.client("lambda", region_name=REGION)
iam = boto3.client("iam")

LAMBDA_NAME = f"{PREFIX}-graph-query-tool"

# Lambda function code
lambda_code = """
import json
import boto3

def handler(event, context):
    """Execute an openCypher query against Neptune Analytics."""
    body = json.loads(event.get("body", "{}")) if isinstance(event.get("body"), str) else event
    query = body.get("query", "")
    graph_id = body.get("graph_id", "")
    
    if not query or not graph_id:
        return {
            "statusCode": 400,
            "body": json.dumps({"error": "query and graph_id are required"})
        }
    
    neptune = boto3.client("neptune-graph")
    response = neptune.execute_query(
        graphIdentifier=graph_id,
        queryString=query,
        language="OPEN_CYPHER"
    )
    results = json.loads(response["payload"].read())
    
    return {
        "statusCode": 200,
        "body": json.dumps(results)
    }
"""

# Package as zip
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w") as zf:
    zf.writestr("lambda_function.py", lambda_code)
zip_buffer.seek(0)

print("\u2713 Lambda package prepared")

In [ ]:
# Create Lambda execution role
LAMBDA_ROLE_NAME = f"{PREFIX}-gateway-lambda-role"
lambda_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{LAMBDA_ROLE_NAME}"

trust_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
})

try:
    iam.get_role(RoleName=LAMBDA_ROLE_NAME)
    print(f"\u2713 Lambda role exists: {LAMBDA_ROLE_NAME}")
except iam.exceptions.NoSuchEntityException:
    iam.create_role(
        RoleName=LAMBDA_ROLE_NAME,
        AssumeRolePolicyDocument=trust_policy,
        Description="Role for OrgIQ gateway Lambda tools"
    )
    iam.attach_role_policy(
        RoleName=LAMBDA_ROLE_NAME,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
    )
    # Add Neptune access
    neptune_policy = json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": ["neptune-graph:ExecuteQuery", "neptune-graph:ReadDataViaQuery"],
            "Resource": "*"
        }]
    })
    iam.put_role_policy(
        RoleName=LAMBDA_ROLE_NAME,
        PolicyName=f"{PREFIX}-neptune-access",
        PolicyDocument=neptune_policy
    )
    tracker.add_iam_role(lambda_role_arn, LAMBDA_ROLE_NAME)
    print(f"\u2713 Lambda role created: {LAMBDA_ROLE_NAME}")
    print("\u231b Waiting 10s for IAM propagation...")
    time.sleep(10)

In [ ]:
# Create or update the Lambda function
try:
    lambda_client.get_function(FunctionName=LAMBDA_NAME)
    lambda_client.update_function_code(
        FunctionName=LAMBDA_NAME,
        ZipFile=zip_buffer.read()
    )
    print(f"\u2713 Lambda function updated: {LAMBDA_NAME}")
except lambda_client.exceptions.ResourceNotFoundException:
    zip_buffer.seek(0)
    lambda_client.create_function(
        FunctionName=LAMBDA_NAME,
        Runtime="python3.12",
        Role=lambda_role_arn,
        Handler="lambda_function.handler",
        Code={"ZipFile": zip_buffer.read()},
        Timeout=30,
        MemorySize=256,
        Tags={"project": PREFIX}
    )
    print(f"\u2713 Lambda function created: {LAMBDA_NAME}")

lambda_arn = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{LAMBDA_NAME}"
print(f"  ARN: {lambda_arn}")

## Step 5: Add Lambda Target to Gateway

In [ ]:
# Register the Lambda as a tool target in the gateway
target_name = "graph-query"

try:
    agentcore.create_gateway_target(
        gatewayId=gateway_id,
        name=target_name,
        description="Execute openCypher queries against the Neptune knowledge graph",
        targetConfiguration={
            "lambdaTarget": {
                "lambdaArn": lambda_arn
            }
        },
        toolSchema={
            "name": "graph_query",
            "description": "Execute an openCypher query against the Neptune Analytics knowledge graph",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "openCypher query to execute"},
                    "graph_id": {"type": "string", "description": "Neptune graph identifier"}
                },
                "required": ["query", "graph_id"]
            }
        }
    )
    print(f"\u2713 Lambda target added: {target_name}")
except Exception as e:
    if "already exists" in str(e).lower() or "conflict" in str(e).lower():
        print(f"\u2713 Target already exists: {target_name}")
    else:
        raise

## Step 6: Add a Second Tool Target (Agent-Based)

In [ ]:
# Add another tool target for document search
target_name_2 = "document-search"

try:
    agentcore.create_gateway_target(
        gatewayId=gateway_id,
        name=target_name_2,
        description="Search organizational documents via Bedrock Knowledge Base",
        targetConfiguration={
            "lambdaTarget": {
                "lambdaArn": lambda_arn  # Reusing same Lambda for demo
            }
        },
        toolSchema={
            "name": "document_search",
            "description": "Search organizational documents using semantic retrieval",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Natural language search query"},
                    "num_results": {"type": "integer", "description": "Number of results", "default": 3}
                },
                "required": ["query"]
            }
        }
    )
    print(f"\u2713 Agent target added: {target_name_2}")
except Exception as e:
    if "already exists" in str(e).lower() or "conflict" in str(e).lower():
        print(f"\u2713 Target already exists: {target_name_2}")
    else:
        raise

## Step 7: List Gateway Tools

Discover available tools through the gateway.

In [ ]:
# List all targets registered with the gateway
targets = agentcore.list_gateway_targets(gatewayId=gateway_id)

print(f"Gateway tools ({len(targets.get('targets', []))}):\n")
for target in targets.get("targets", []):
    print(f"  Tool: {target['name']}")
    print(f"    Description: {target.get('description', 'N/A')}")
    print(f"    Type: {target.get('targetConfiguration', {}).keys()}")
    print()

## Step 8: Invoke Tool via Gateway

Call a tool through the MCP gateway.

In [ ]:
# Invoke the graph query tool via gateway
GRAPH_ID = config["NEPTUNE_GRAPH_ID"]

response = agentcore.invoke_gateway_tool(
    gatewayId=gateway_id,
    targetName="graph-query",
    toolInput={
        "query": "MATCH (p:Person) RETURN p.name AS name, p.title AS title LIMIT 3",
        "graph_id": GRAPH_ID
    }
)

result = json.loads(response["body"])
print("Gateway tool invocation result:")
print(json.dumps(result, indent=2))
print("\n\u2713 Tool invocation via gateway working")

## Step 9: Use Gateway Tools in a Strands Agent

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

# Create a tool that wraps gateway invocation
@tool
def gateway_graph_query(query: str) -> str:
    """Execute a graph query via the MCP Gateway.
    
    Args:
        query: openCypher query to execute against the knowledge graph
    """
    response = agentcore.invoke_gateway_tool(
        gatewayId=gateway_id,
        targetName="graph-query",
        toolInput={"query": query, "graph_id": GRAPH_ID}
    )
    return response["body"]

# Create agent with gateway-backed tools
model = BedrockModel(model_id=config["MODEL_ID"], region_name=REGION)
gateway_agent = Agent(
    model=model,
    system_prompt="You are an org search assistant. Use gateway_graph_query to search the knowledge graph with openCypher.",
    tools=[gateway_graph_query]
)

response = gateway_agent("Find all people and their expertise topics.")
print("Agent response (via gateway):")
print(response)

## Step 10: Gateway Configuration Details

In [ ]:
# Get gateway details
gateway_info = agentcore.get_gateway(gatewayId=gateway_id)

print(f"Gateway: {gateway_info['name']}")
print(f"  ID: {gateway_info['id']}")
print(f"  Protocol: {gateway_info.get('protocolType', 'MCP')}")
print(f"  Status: {gateway_info.get('status', 'N/A')}")
print(f"  Endpoint: {gateway_info.get('endpoint', 'N/A')}")

print("\n\u2713 Gateway configuration verified")

## Step 11: Save Configuration

In [ ]:
config["GATEWAY_ID"] = gateway_id
Path("config.json").write_text(json.dumps(config, indent=2))

print(f"\u2713 Config saved")
print(f"  GATEWAY_ID: {gateway_id}")

## Summary

- [x] MCP Gateway created for centralized tool management
- [x] Lambda function deployed as tool backend
- [x] Lambda target registered with tool schema
- [x] Document search target registered
- [x] Gateway tool discovery (list targets) working
- [x] Tool invocation via gateway working
- [x] Strands agent using gateway-backed tools

**Next:** [09-agentcore-policy.ipynb](09-agentcore-policy.ipynb) — Create policy engine with Cedar policies for access control